## 基本环境 · Basic setup

首次打开运行下面 3 个 cell。它们做的事:
1. 把工作目录切到 `solutions/` (这样 `from attention.mha import ...` 这种导入能直接生效)。
2. 启用 `autoreload`，编辑 .py 文件保存后 notebook 里立刻可用，不用重启 kernel。
3. 设 `LAYERNORM_TYPE=torch`，避免 CUDA-only 算子的 import 失败。

First time you open the notebook, run the 3 cells below: cd into `solutions/`, turn on autoreload, force the pure-PyTorch LayerNorm path.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) != 'solutions':
    if os.path.isdir('solutions'):
        os.chdir('solutions')
    else:
        # already inside a chapter folder — climb out
        while os.path.basename(os.getcwd()) != 'solutions' and os.getcwd() != '/':
            os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
os.environ.setdefault('LAYERNORM_TYPE', 'torch')
print('cwd =', os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
# Folder where the pairformer chapter's reference .pt files live
control_folder = 'pairformer/control_values'
assert os.path.isdir(control_folder), f'missing {control_folder}'

# 第 2 章 · Pairformer

Pairformer 是 AF3 的核心主干 (替代 AF2 的 Evoformer)。它由若干**三角更新**算子组合而成，让 pair 表示 $z_{ij}$ 之间的关系满足[三角不等式](https://en.wikipedia.org/wiki/Triangle_inequality)。

本章涉及的模块 (全部位于 `pairformer/`):

| 文件 | 类 | 作用 |
|---|---|---|
| `triangle_ops.py` | `OuterProductMean` | 算法 10 — MSA → pair |
| `triangle.py` | `TriangleMultiplication{Outgoing,Incoming}` | 算法 11 / 12 |
| `triangle.py` | `TriangleAttention` | 算法 13 / 14 |
| `msa_stack.py` | `MSAPairWeightedAveraging` | MSAModule 内 pair → MSA 的反向通道 |
| `pair_stack.py` | `PairformerBlock` | 算法 17 单块 (含全部三角操作) |

## 2.1 OuterProductMean (算法 10)

打开 `pairformer/triangle_ops.py`，填 `OuterProductMean.__init__`、`_opm`、`_forward` 三处 TODO。

OPM 把 MSA 张量沿序列维做外积取均值，把 MSA 信息汇入 pair 通道。

In [ ]:
from pairformer.triangle_ops import OuterProductMean
from pairformer.control_values.pairformer_checks import (
    c_m, c_z, c_hidden, no_heads_pair, test_inputs,
    test_module_shape, test_module_method, test_module_forward,
)

opm = OuterProductMean(c_m=c_m, c_z=c_z, c_hidden=c_hidden)
test_module_shape(opm, 'outer_product_mean', control_folder)
test_module_method(
    opm, 'outer_product_mean',
    inputs=(test_inputs['m'], test_inputs['msa_mask']),
    output_names='out',
    control_folder=control_folder,
    method=lambda m, mask: opm(m, mask=mask),
)
print('OuterProductMean ✓')

## 2.2 TriangleMultiplication (算法 11 / 12)

打开 `pairformer/triangle.py`，填 `BaseTriangleMultiplicativeUpdate.__init__`、`TriangleMultiplicativeUpdate.__init__`、`_combine_projections`、`forward` 几处 TODO。

外向 (outgoing) 与内向 (incoming) 走同一个类，只是子类用 `partialmethod` 固定 `_outgoing` 标志。

In [ ]:
from pairformer.triangle import (
    TriangleMultiplicationOutgoing, TriangleMultiplicationIncoming,
)

for variant, cls in [
    ('triangle_mul_out', TriangleMultiplicationOutgoing),
    ('triangle_mul_in',  TriangleMultiplicationIncoming),
]:
    mod = cls(c_z=c_z, c_hidden=c_hidden)
    test_module_shape(mod, variant, control_folder)
    test_module_method(
        mod, variant,
        inputs=(test_inputs['z'], test_inputs['pair_mask']),
        output_names='out',
        control_folder=control_folder,
        method=lambda z, pm, mod=mod: mod(z, mask=pm),
    )
print('TriangleMultiplication (outgoing + incoming) ✓')

## 2.3 TriangleAttention (算法 13 / 14)

在 `pairformer/triangle.py` 里继续，填 `TriangleAttention.__init__` 与 `.forward`。

三角自注意力对 pair 张量沿一条 residue 轴跑多头注意力，把另一条 residue 轴的信息作为 bias。起始节点 (starting=True) 与终止节点 (ending) 共用同一个类，end 模式通过物理转置实现。

In [ ]:
from pairformer.triangle import TriangleAttention

tri_att = TriangleAttention(
    c_in=c_z, c_hidden=c_hidden, no_heads=no_heads_pair, starting=True,
)
test_module_shape(tri_att, 'triangle_attention_start', control_folder)
test_module_method(
    tri_att, 'triangle_attention_start',
    inputs=(test_inputs['z'], test_inputs['pair_mask']),
    output_names='out',
    control_folder=control_folder,
    method=lambda z, pm: tri_att(z, mask=pm),
)
print('TriangleAttention ✓')

## 2.4 MSAPairWeightedAveraging

打开 `pairformer/msa_stack.py`，填 `MSAPairWeightedAveraging.__init__` 与 `.forward`。

MSAModule 内由 pair 张量反过来更新 MSA 通道用的就是这一块——pair 的每对 (i, j) 产生权重，沿 j 轴对 MSA 的 value 加权平均。

In [ ]:
from pairformer.msa_stack import MSAPairWeightedAveraging

mpwa = MSAPairWeightedAveraging(c_m=c_m, c=c_hidden, c_z=c_z, n_heads=no_heads_pair)
test_module_shape(mpwa, 'msa_pair_weighted_avg', control_folder)
test_module_forward(
    mpwa, 'msa_pair_weighted_avg',
    inputs=(test_inputs['m'], test_inputs['z']),
    output_names='out',
    control_folder=control_folder,
)
print('MSAPairWeightedAveraging ✓')

## 2.5 PairformerBlock (算法 17 — 一整块)

打开 `pairformer/pair_stack.py`，把 `PairformerBlock.__init__` 与 `.forward` 两处 TODO 填好。

一个 block 把前面 4 个三角操作 + pair transition 串起来，更新 pair 表示。若 `c_s > 0` 还会再叠一次 AttentionPairBias + Transition 更新 single 表示。

本测试用 `c_s=0` 简化路径。

In [ ]:
from pairformer.pair_stack import PairformerBlock

block = PairformerBlock(
    n_heads=no_heads_pair,
    c_z=c_z, c_s=0,
    c_hidden_mul=c_hidden,
    c_hidden_pair_att=c_hidden,
    no_heads_pair=no_heads_pair,
    num_intermediate_factor=2,
    dropout=0.0,
)
for sub in (block.tri_att_start.mha, block.tri_att_end.mha):
    if hasattr(sub, 'use_efficient_implementation'):
        sub.use_efficient_implementation = False

test_module_shape(block, 'pairformer_block_no_single', control_folder)
test_module_method(
    block, 'pairformer_block_no_single',
    inputs=(None, test_inputs['z'], test_inputs['pair_mask']),
    output_names='z_out',
    control_folder=control_folder,
    method=lambda s, z, pm: block(s, z, pair_mask=pm)[1],
)
print('PairformerBlock ✓')

## 章节小结

本章完成后你已经能用三角更新算子把 pair 表示从 MSA 中提炼出来，并能复用上一章实现的 attention / transition 把它们组合成一个完整的 PairformerBlock。把若干个 block 堆叠起来就是 Algorithm 17 的 PairformerStack。